In [66]:
import pandas as pd
import os

current_dir = os.path.abspath('.')
root_dir = os.path.dirname(current_dir) 
file_path = os.path.join(root_dir, 'data/raw', 'default of credit card clients.csv')
    
# Charger le CSV dans un DataFrame
df = pd.read_csv(file_path)

## Nettoyage suite à Audit

In [67]:
# Afficher les répartitions avant nettoyage
print("\n=== RÉPARTITION AVANT NETTOYAGE ===")
print("MARRIAGE :")
marriage_counts = df['MARRIAGE'].value_counts().sort_index()
for value, count in marriage_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - Valeur {value} : {count} sur {len(df)} ({percentage:.2f}%)")

print("\nEDUCATION :")
education_counts = df['EDUCATION'].value_counts().sort_index()
for value, count in education_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - Valeur {value} : {count} sur {len(df)} ({percentage:.2f}%)")

# Nettoyage de la colonne MARRIAGE
print("=== NETTOYAGE DE LA COLONNE MARRIAGE ===")
print(f"Valeurs originales dans MARRIAGE : {df['MARRIAGE'].unique().tolist()}")

# Remplacer les valeurs en dehors de [1, 2, 3] par 2 (autres)
df['MARRIAGE'] = df['MARRIAGE'].replace([x for x in df['MARRIAGE'].unique() if x not in [1, 2, 3]], 3)

print(f"Valeurs après nettoyage : {df['MARRIAGE'].unique().tolist()}")
print("Nettoyage de MARRIAGE terminé")

# Nettoyage de la colonne EDUCATION
print("\n=== NETTOYAGE DE LA COLONNE EDUCATION ===")
print(f"Valeurs originales dans EDUCATION : {df['EDUCATION'].unique().tolist()}")

# Remplacer les valeurs en dehors de [1, 2, 3, 4] par 4 (autres)
df['EDUCATION'] = df['EDUCATION'].replace([x for x in df['EDUCATION'].unique() if x not in [1, 2, 3, 4]], 4)

print(f"Valeurs après nettoyage : {df['EDUCATION'].unique().tolist()}")
print("Nettoyage de EDUCATION terminé")

# Afficher les répartitions après nettoyage
print("\n=== RÉPARTITION APRÈS NETTOYAGE ===")
print("MARRIAGE :")
marriage_counts = df['MARRIAGE'].value_counts().sort_index()
for value, count in marriage_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - Valeur {value} : {count} sur {len(df)} ({percentage:.2f}%)")

print("\nEDUCATION :")
education_counts = df['EDUCATION'].value_counts().sort_index()
for value, count in education_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - Valeur {value} : {count} sur {len(df)} ({percentage:.2f}%)")


=== RÉPARTITION AVANT NETTOYAGE ===
MARRIAGE :
  - Valeur 0 : 54 sur 30000 (0.18%)
  - Valeur 1 : 13659 sur 30000 (45.53%)
  - Valeur 2 : 15964 sur 30000 (53.21%)
  - Valeur 3 : 323 sur 30000 (1.08%)

EDUCATION :
  - Valeur 0 : 14 sur 30000 (0.05%)
  - Valeur 1 : 10585 sur 30000 (35.28%)
  - Valeur 2 : 14030 sur 30000 (46.77%)
  - Valeur 3 : 4917 sur 30000 (16.39%)
  - Valeur 4 : 123 sur 30000 (0.41%)
  - Valeur 5 : 280 sur 30000 (0.93%)
  - Valeur 6 : 51 sur 30000 (0.17%)
=== NETTOYAGE DE LA COLONNE MARRIAGE ===
Valeurs originales dans MARRIAGE : [1, 2, 3, 0]
Valeurs après nettoyage : [1, 2, 3]
Nettoyage de MARRIAGE terminé

=== NETTOYAGE DE LA COLONNE EDUCATION ===
Valeurs originales dans EDUCATION : [2, 1, 3, 5, 4, 6, 0]
Valeurs après nettoyage : [2, 1, 3, 4]
Nettoyage de EDUCATION terminé

=== RÉPARTITION APRÈS NETTOYAGE ===
MARRIAGE :
  - Valeur 1 : 13659 sur 30000 (45.53%)
  - Valeur 2 : 15964 sur 30000 (53.21%)
  - Valeur 3 : 377 sur 30000 (1.26%)

EDUCATION :
  - Valeur 1 : 10

In [68]:
import os
from pathlib import Path

def save_cleaned_dataset(df, filename='cleaned_creditcard.csv'):
    """
    Sauvegarde le DataFrame nettoyé dans le dossier data à la racine
    
    Parameters:
    df (pd.DataFrame): DataFrame nettoyé à sauvegarder
    filename (str): Nom du fichier de sortie
    """
    
    # Construire le chemin vers le dossier data à la racine
    # On utilise Path pour une gestion plus robuste des chemins
    current_path = Path.cwd()  # Chemin du dossier courant
    root_dir = current_path.parent  # Dossier racine (un niveau au-dessus de src)
    data_dir = root_dir / 'data'  # Dossier data
    
    # Vérifier que le dossier data existe, sinon le créer
    if not data_dir.exists():
        data_dir.mkdir(parents=True, exist_ok=True)
        print(f"Dossier {data_dir} créé")
    
    # Construire le chemin complet du fichier
    file_path = data_dir / filename
    
    # Sauvegarder le DataFrame en CSV
    df.to_csv(file_path, index=False, encoding="utf-8")
    
    print(f"DataFrame sauvegardé dans : {file_path}")
    print(f"Nombre de lignes : {len(df)}")
    print(f"Nombre de colonnes : {len(df.columns)}")

In [69]:
# enregistrement du CSV pret pour ingestion dans la BDD
save_cleaned_dataset(df=df, filename = 'creditcard_pret_ingestion.csv')

DataFrame sauvegardé dans : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\data\creditcard_pret_ingestion.csv
Nombre de lignes : 30000
Nombre de colonnes : 25


## Nettoyages décidés après EDA_lab
- 4 lignes avec PAY_AMTn > 1 000 000
- 795 comptes inactifs : PAY_AMTn == 0 et BILL_AMTn == 0 sur les 6 mois
- 48 comptes inactifs : PAY_AMTn == 0 et BILL_AMTn < 0 et stable sur les 6 mois
- Correction des PAY_n = 1 : PAY_n = PAY_(n+1) si BILL_AMT(n+1) <= 0
- le client 6783 a une codification PAY = 1 sur 4 mois alors qu'il paie chaque mois -> remettre sa codification en 0

In [70]:
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_cols = [f'PAY_AMT{i}' for i in range(1, 7)]

# 1. Filtre sur les paiements extrêmes (> 1 000 000)
mask_extreme_pay = (df[pay_cols] > 1000000).any(axis=1)

# 2. Filtre sur les inactifs stricts (BILL == 0 et PAY == 0)
mask_strictly_zero = (df[bill_cols] == 0).all(axis=1) & (
    df[pay_cols] == 0
).all(axis=1)

# 3. Filtre sur les inactifs à solde négatif STABLE sans paiement
mask_no_payments = (df[pay_cols] == 0).all(axis=1)
mask_all_negative = (df[bill_cols] < 0).all(axis=1)
mask_all_equal = (
    df[bill_cols].nunique(axis=1) == 1
)  # Vérifie la stabilité exacte des 6 montants

mask_neg_dormant = mask_no_payments & mask_all_negative & mask_all_equal

# Affichage du détail par catégorie
print("=== DÉTAIL DES SUPPRESSIONS ===")
print(f"1. Paiements extrêmes (> 1 000 000)                     : {mask_extreme_pay.sum()}")
print(f"2. Inactifs stricts (BILL = 0 & PAY = 0)                 : {mask_strictly_zero.sum()}")
print(f"3. Inactifs solde négatif STABLE (BILL < 0 égal & PAY=0) : {mask_neg_dormant.sum()}")

# Agrégation des règles d'exclusion
mask_to_remove = mask_extreme_pay | mask_strictly_zero | mask_neg_dormant

# Application du nettoyage
df_clean = df[~mask_to_remove].copy()

print("\n=== BILAN GLOBAL ===")
print(f"Total supprimé    : {mask_to_remove.sum()} lignes ({mask_to_remove.mean():.2%})")
print(f"Lignes conservées : {len(df_clean)} / {len(df)}")

=== DÉTAIL DES SUPPRESSIONS ===
1. Paiements extrêmes (> 1 000 000)                     : 4
2. Inactifs stricts (BILL = 0 & PAY = 0)                 : 795
3. Inactifs solde négatif STABLE (BILL < 0 égal & PAY=0) : 48

=== BILAN GLOBAL ===
Total supprimé    : 847 lignes (2.82%)
Lignes conservées : 29153 / 30000


In [71]:
# Définir la condition : PAY_2 = 2, PAY_3 = -2 et BILL_AMT3 <= 0
condition = (df_clean['PAY_2'] == 2) & (df_clean['PAY_3'] == -2) & (df_clean['BILL_AMT3'] <= 0)

# Compter le nombre de corrections à appliquer
nombre_corrections = condition.sum()

print(f"Nombre de corrections à appliquer : {nombre_corrections}")

# Appliquer la correction : PAY_2 = 2 devient PAY_2 = -2
df_clean.loc[condition, 'PAY_2'] = -2

# Vérifier que les corrections ont été appliquées
print("\nVérification des corrections :")
lignes_corrigees = df_clean[condition]
print(f"Nombre de lignes où PAY_2 = -2 après correction : {(lignes_corrigees['PAY_2'] == -2).sum()}")



Nombre de corrections à appliquer : 53

Vérification des corrections :
Nombre de lignes où PAY_2 = -2 après correction : 53


In [72]:
# Correction de PAY_n = PAY_(n+1) si (BILL_AMT(n+1) <= 0 et PAY_n = 1)
# En partant de PAY_5 vers PAY_1

nb_corrections = 0

for i in range(5, 0, -1):  # De PAY_5 à PAY_1
    col_pay = f'PAY_{i}'
    col_pay_suivant = f'PAY_{i+1}'
    col_bill = f'BILL_AMT{i+1}'
    
    if col_pay in df.columns and col_pay_suivant in df.columns and col_bill in df.columns:
        # Condition : BILL_AMT(n+1) <= 0 et PAY_n = 1
        condition = (df[col_bill] <= 0) & (df[col_pay] == 1)
        
        # Compter les corrections
        nb_corr = condition.sum()
        nb_corrections += nb_corr
        
        if nb_corr > 0:
            print(f"Correction appliquée sur {nb_corr} lignes : {col_pay} devient {col_pay_suivant}")
            # Appliquer la correction
            df.loc[condition, col_pay] = df.loc[condition, col_pay_suivant]

print(f"Nombre total de corrections appliquées : {nb_corrections}")

Correction appliquée sur 12 lignes : PAY_2 devient PAY_3
Correction appliquée sur 1175 lignes : PAY_1 devient PAY_2
Nombre total de corrections appliquées : 1187


In [73]:
# Vérifier les valeurs actuelles avant modification
print("Valeurs avant modification :")
ligne_6783 = df[df['ID'] == 6783]
print(ligne_6783[['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4']])

# Modifier les colonnes PAY_1 à PAY_4 de 1 à 0 pour l'ID 6783
df.loc[df['ID'] == 6783, ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4']] = 0

# Vérifier les nouvelles valeurs
print("\nValeurs après modification :")
ligne_6783_apres = df[df['ID'] == 6783]
print(ligne_6783_apres[['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4']])

print("\nModification appliquée avec succès !")

Valeurs avant modification :
      PAY_1  PAY_2  PAY_3  PAY_4
6782      1      1      1      1

Valeurs après modification :
      PAY_1  PAY_2  PAY_3  PAY_4
6782      0      0      0      0

Modification appliquée avec succès !


In [74]:
save_cleaned_dataset(df_clean, 'cleaned_creditcard.csv')

DataFrame sauvegardé dans : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\data\cleaned_creditcard.csv
Nombre de lignes : 29153
Nombre de colonnes : 25
